In [3]:
import json
from tqdm import tqdm

def read_game(filename):
    with open(filename, 'r') as f:
        game_set_data = json.load(f)

    filtered_games = {}
    unfinished = 0
    aborted = 0
    lobby = 0
    other = 0
    too_few_players = 0
    too_few_rounds = 0
    bots = 0
    no_chat = 0

    for game_id in tqdm(game_set_data):
        game_data = game_set_data[game_id]

        public_info = game_data['publicInfo']
        private_info = game_data['privateInfo']
        status = public_info['gameStatus'] 

        if status != 'finished':
            unfinished += 1
            if status == 'lobby':
                lobby +=1
            elif status == 'aborted':
                aborted += 1
            else:
                other += 1

            continue

        num_players = public_info['numPlayers']

        if num_players < 5:
            too_few_players += 1
            continue

        round_num = public_info['roundNum']

        if round_num < 5:
            too_few_rounds += 1
            continue

        bot_found = False

        players = private_info['players']
        for player in players:
            experience = players[player]['experience']
            if experience == "advanced":
                bot_found = True
                break

        if bot_found:
            bots += 1
            continue

        if public_info['gameParams']['chatType'] != 'direct':
            no_chat += 1
            continue

        filtered_games[game_id] = game_data

    print(f"Total filtered games: {len(filtered_games)}")
    print (f"Unfinished: {unfinished}  Aborted: {aborted}  Lobby:  {lobby}  Other: {other}")
    print(f"Too few players: {too_few_players}  Too few rounds:  {too_few_rounds}  Bots: {bots}  No chat: {no_chat}")

    save_file = filename.replace('.json', '_filtered.json')
    with open(save_file, 'w') as f:
        json.dump(filtered_games, f, indent=2)  

read_game('preprod.json')

100%|██████████| 5251/5251 [00:00<00:00, 768762.97it/s]


Total filtered games: 156
Unfinished: 1311  Aborted: 0  Lobby:  1243  Other: 68
Too few players: 2369  Too few rounds:  658  Bots: 644  No chat: 113


In [5]:
from pathlib import Path
import json
import tqdm

def extract_chat_info(
    input_file='preprod_filtered.json',
    output_folder='chat_info'
):
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    with open(input_file, 'r') as f:
        games = json.load(f)

    for game_id, game_data in tqdm.tqdm(games.items()):
        chat_info = game_data.get('chatInfo', {})

        output_file = output_folder / f'{game_id}.json'
        with open(output_file, 'w') as f:
            json.dump(chat_info, f, indent=2)

    print(f'Created {len(games)} chat files in "{output_folder}"')

extract_chat_info()

100%|██████████| 156/156 [00:00<00:00, 427.38it/s]

Created 156 chat files in "chat_info"
